# P5 r2 — Drive 입력 · 안전 재개 · CPU probe 전용

기존 GPU 추출 결과와 완료 probe를 이어갑니다. **GPU 추출 셀은 없습니다.** 기존 노트북에서 실행 중인 probe/복사 셀을 중단하고 이 노트북만 실행하세요. 같은 Drive 폴더에 두 실행을 동시에 쓰지 마세요.

1. 기존 `v1_4_p5_bundle_r2.zip`을 Google Drive `내 드라이브/boolean_interp_v1_4/`에 한 번 올립니다. 새 bundle은 필요 없습니다. 이미 다른 위치에 올렸으면 첫 셀의 경로만 바꿉니다.
2. Drive mount → bundle 로컬 복사·checksum → 의존성 → 안전 복구 → CPU probe 순서입니다.
3. CPU 런타임으로 충분합니다. 런타임 변경 시 로컬 파일은 사라질 수 있으나 Drive 저장 결과를 재개합니다.
4. 이번 변경은 전달·복구 절차만 바꿉니다. 동결된 r2 실험 코드/라벨/선택 규칙을 변경하지 않습니다.


In [ ]:
from google.colab import drive, files
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, zipfile
# 이미 mount되어 있다는 문구는 오류가 아닙니다.
drive.mount('/content/drive')
BUNDLE_ON_DRIVE=Path('/content/drive/MyDrive/boolean_interp_v1_4/v1_4_p5_bundle_r2.zip')
# 내 드라이브 최상위에 올린 경우 자동으로 찾습니다.
if not BUNDLE_ON_DRIVE.exists():
    alternative=Path('/content/drive/MyDrive/v1_4_p5_bundle_r2.zip')
    if alternative.exists():BUNDLE_ON_DRIVE=alternative
assert BUNDLE_ON_DRIVE.is_file(), 'BUNDLE_ON_DRIVE를 Drive에 올린 r2 ZIP 경로로 수정하세요.'
ROOT=Path('/content/boolean_interp')
OUT=ROOT/'experiment_v1_4/runs/p5_r2'
PERSIST=Path('/content/drive/MyDrive/boolean_interp_v1_4/P5_r2')
assert (PERSIST/'contract.json').exists(), '기존 추출 결과의 P5_r2 Drive 경로를 확인하세요.'
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''):h.update(b)
    return h.hexdigest()
print('Drive bundle:',BUNDLE_ON_DRIVE)
print('기존 결과:',PERSIST)


In [ ]:
EXPECTED_BUNDLE_SHA='afe16d82028fbefd603b601e17a68ab4719a83b1625d5f3099191c00d72991a3'
ROOT.mkdir(parents=True,exist_ok=True)
local_bundle=Path('/content/v1_4_p5_bundle_r2.zip')
if not local_bundle.exists() or sha(local_bundle)!=EXPECTED_BUNDLE_SHA:
    temp=local_bundle.with_suffix('.zip.copy.tmp')
    shutil.copyfile(BUNDLE_ON_DRIVE,temp)
    assert sha(temp)==EXPECTED_BUNDLE_SHA, 'Drive bundle checksum 불일치: r2 원본 ZIP을 확인하세요.'
    temp.replace(local_bundle)
input_backup=PERSIST.parent/'P5_input_backups'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
with zipfile.ZipFile(local_bundle) as z:
    inventory=json.loads(z.read('bundle_manifest.json'))['files']
    assert len(z.namelist())==len(set(z.namelist()))
    assert set(z.namelist())==set(inventory)|{'bundle_manifest.json'}
    for name,digest in inventory.items():
        dest=ROOT/name
        assert dest.resolve().is_relative_to(ROOT.resolve()), '잘못된 ZIP 경로'
        if dest.exists() and sha(dest)==digest:continue
        if dest.exists():
            backup=input_backup/name;backup.parent.mkdir(parents=True,exist_ok=True)
            shutil.copyfile(dest,backup);assert sha(dest)==sha(backup)
        dest.parent.mkdir(parents=True,exist_ok=True)
        temp=dest.with_suffix(dest.suffix+'.input.tmp')
        with z.open(name) as src,temp.open('wb') as dst:shutil.copyfileobj(src,dst,1024*1024)
        assert sha(temp)==digest, '내부 checksum 불일치: '+name
        temp.replace(dest)
os.chdir(ROOT)
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
print('Drive → 로컬 입력 검증 완료. 이미 일치하는 파일은 재사용했습니다.')


## 의존성 준비
기존 r2의 실제 패키지 lock을 사용합니다. CPU 계산은 GPU를 요구하지 않습니다. 새 런타임에서는 의존성 다운로드가 한 번 필요할 수 있습니다. 패키지 변경 후 Colab이 재시작을 요구하면 재시작하고 첫 셀부터 다시 실행하세요.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r',str(ROOT/'experiment_v1_4/frozen_test_r1/requirements-primary.lock.txt'),'--extra-index-url','https://download.pytorch.org/whl/cu128'],check=True)

## 영속 저장·안전 재개

라벨은 동결 corpus에서 독립 재생성하고, cache marker가 요구하는 label hash와 일치하는지 먼저 확인합니다. 손상·공백 차이 등으로 다른 기존 라벨은 `P5_resume_backups/실행시각/`에 원본과 hash를 보존한 뒤 복구합니다. 해당 원인은 같은 폴더의 `repair.jsonl`에 기록됩니다.

Cache는 marker SHA256과 일치하는 복사본을 사용합니다. 완료 probe는 task/config/구조를 확인해 재사용합니다. **양쪽 모두 유효한데 서로 다른 결과이면 임의로 선택하지 않고 백업 후 중단합니다.** 이 경우 오류 메시지와 repair.jsonl을 전달하세요. Drive 파일을 매번 읽고 hash를 계산하므로 이 셀도 시간이 걸릴 수 있지만 이미 동일한 로컬 cache는 다시 복사하지 않습니다.


In [ ]:
helper_path=Path('/content/p5_cpu_resume.py')
helper_path.write_text('"""P5 CPU delivery repair. Does not modify frozen inference/probe source or choices."""\nfrom datetime import datetime, timezone\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport shutil\n\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\n\ndef canonical(value):\n    return (json.dumps(value,indent=2,ensure_ascii=False,allow_nan=False)+\'\\n\').encode()\n\n\nclass Repair:\n    def __init__(self, local, drive, backup):\n        self.local=Path(local);self.drive=Path(drive);self.backup=Path(backup)\n        self.log=[]\n        self.backup.mkdir(parents=True,exist_ok=True)\n\n    def record(self, **row):\n        self.log.append(row)\n        with (self.backup/\'repair.jsonl\').open(\'a\') as f:f.write(json.dumps(row)+\'\\n\')\n\n    def preserve(self,path):\n        if not path.exists():return\n        side=\'local\' if path.is_relative_to(self.local) else \'drive\'\n        base=self.local if side==\'local\' else self.drive\n        dst=self.backup/side/path.relative_to(base)\n        dst.parent.mkdir(parents=True,exist_ok=True)\n        if dst.exists():\n            if sha(dst)!=sha(path):raise ValueError(\'Backup conflict: \'+str(dst))\n        else:\n            shutil.copyfile(path,dst)\n            if sha(dst)!=sha(path):raise ValueError(\'Backup verification failed\')\n        self.record(action=\'preserved\',side=side,path=str(path),sha256=sha(dst))\n\n    def put(self,dest,source=None,payload=None,reason=\'verified copy\'):\n        expected=sha(source) if source is not None else hashlib.sha256(payload).hexdigest()\n        if dest.exists() and sha(dest)==expected:return\n        self.preserve(dest)\n        dest.parent.mkdir(parents=True,exist_ok=True)\n        tmp=dest.with_suffix(dest.suffix+\'.resume.tmp\')\n        if source is not None:\n            shutil.copyfile(source,tmp)\n        else:\n            with tmp.open(\'wb\') as f:f.write(payload);f.flush();os.fsync(f.fileno())\n        if sha(tmp)!=expected:raise ValueError(\'Incomplete copy: \'+str(dest))\n        tmp.replace(dest)\n        self.record(action=\'restored\',path=str(dest),sha256=expected,reason=reason)\n\n    def reconcile(self,relative,validator):\n        a=self.local/relative;b=self.drive/relative\n        def valid(p):\n            if not p.exists():return False\n            try:return bool(validator(p))\n            except (ValueError,KeyError,TypeError,OSError):return False\n        va,vb=valid(a),valid(b)\n        if va and vb:\n            if sha(a)!=sha(b):\n                # Whitespace-only JSON differences do not change saved experiment results.\n                same=False\n                if a.suffix==\'.json\':\n                    try:same=json.loads(a.read_text())==json.loads(b.read_text())\n                    except ValueError:pass\n                if not same:\n                    self.preserve(a);self.preserve(b)\n                    raise ValueError(\'Two different valid results; preserved both, no automatic selection: \'+str(relative))\n                self.put(a,source=b,reason=\'equal JSON values; normalize to Drive bytes\')\n        elif vb:self.put(a,source=b)\n        elif va:self.put(b,source=a,reason=\'valid local copy repairs missing/invalid Drive copy\')\n        else:\n            self.preserve(a);self.preserve(b)\n            raise ValueError(\'No verified copy: \'+str(relative))\n\n\ndef validate_probe(path,config,ch):\n    x=json.loads(path.read_text());task=config[\'tasks\'][path.stem]\n    if x[\'config_sha256\']!=ch or x[\'task_key\']!=task[\'key\']:return False\n    if x[\'bootstrap_seed\']!=task[\'bootstrap_seed\'] or x.get(\'shuffle_seed\')!=task.get(\'shuffle_seed\'):return False\n    r=x[\'result\']\n    if r[\'status\'] in (\'NA\',\'failed\'):\n        return bool(r.get(\'reason\')) # Preserve recorded failures; never silently replace by success.\n    if r[\'status\']!=\'passed\' or not r.get(\'trace\') or not r.get(\'selected\') or not r.get(\'evaluation\'):return False\n    return set(r[\'selected\'])==set(r[\'evaluation\']) and all(\'coefficients\' in f for f in r[\'selected\'].values())\n\n\ndef restore(root,local,drive):\n    from interp_v1_4.p5 import verify,load_split,support_table,CONTRACT\n    root=Path(root);local=Path(local);drive=Path(drive)\n    if local.resolve()==drive.resolve():raise ValueError(\'Separate local and Drive folders required\')\n    config=verify(root);ch=sha(root/CONTRACT)\n    if not (drive/\'contract.json\').exists():raise ValueError(\'Original P5_r2 Drive folder not found\')\n    # A different experiment root is never repaired into the requested experiment.\n    if json.loads((drive/\'contract.json\').read_text())!=config:raise ValueError(\'Drive contains another contract\')\n    stamp=datetime.now(timezone.utc).strftime(\'%Y%m%dT%H%M%S%f\')\n    repair=Repair(local,drive,drive.parent/\'P5_resume_backups\'/stamp)\n    repair.put(local/\'contract.json\',payload=canonical(config),reason=\'frozen input contract\')\n    repair.put(drive/\'contract.json\',payload=canonical(config),reason=\'frozen input contract\')\n    print(\'Backup/log:\',repair.backup,flush=True)\n    # Labels can be independently reconstructed from frozen corpus, without any inference.\n    for split in config[\'quotas\']:\n        _,rows=load_split(root,config,split);payload=canonical(rows);expected=hashlib.sha256(payload).hexdigest()\n        for base in (local,drive):\n            for marker in (base/\'cache\').glob(f\'*/{split}/*.json\'):\n                try:identity=json.loads(marker.read_text())[\'identity\']\n                except (ValueError,KeyError):continue\n                if identity.get(\'config_sha256\')==ch and identity.get(\'labels_sha256\')!=expected:\n                    raise ValueError(\'Cache expects different labels; no repair performed for \'+split)\n        for base in (local,drive):\n            path=base/\'labels\'/f\'{split}.json\'\n            if path.exists() and sha(path)!=expected:\n                try:equal=json.loads(path.read_text())==rows\n                except ValueError:equal=False\n                repair.record(action=\'label_mismatch\',path=str(path),actual_sha256=sha(path),expected_sha256=expected,\n                              semantic_equal=equal,bytes=path.stat().st_size)\n            repair.put(path,payload=payload,reason=\'independent frozen corpus READ label reconstruction\')\n            repair.put(base/\'labels\'/f\'{split}.support.json\',payload=canonical(support_table(rows)),reason=\'frozen label support\')\n        print(\'Labels verified:\',split,len(rows),flush=True)\n    def inventory(base):return {p.relative_to(base) for p in base.rglob(\'*\') if p.is_file() and not p.name.endswith(\'.tmp\')}\n    names=inventory(local)|inventory(drive)\n    markers=sorted(n for n in names if n.parts[0]==\'cache\' and n.suffix==\'.json\')\n    # Process checksum markers before the arrays they authenticate.\n    for rel in markers:\n        def marker_ok(p):\n            x=json.loads(p.read_text());i=x[\'identity\'];seed=i[\'lm_seed\'];kind=rel.parts[1].split(\'_\',1)[1];split=rel.parts[2]\n            m=next(m for m in config[\'models\'] if m[\'lm_seed\']==seed)\n            cp=m[\'checkpoint\'] if kind==\'trained\' else m[\'init_checkpoint\']\n            return (i[\'config_sha256\']==ch and i[\'split\']==split and i[\'position_type\']==\'READ\'\n                and i[\'checkpoint_sha256\']==config[\'files\'][cp]\n                and i[\'labels_sha256\']==sha(local/\'labels\'/f\'{split}.json\') and len(x[\'sha256\'])==64)\n        repair.reconcile(rel,marker_ok)\n        expected=json.loads((local/rel).read_text())[\'sha256\']\n        repair.reconcile(rel.with_suffix(\'.npz\'),lambda p:sha(p)==expected)\n    print(\'Cache chunks synchronized:\',len(markers),flush=True)\n    # Never adopt an uncommitted array with no complete checksum marker.\n    for rel in sorted(names):\n        if rel.parts[0] in (\'cache\',\'labels\') or str(rel)==\'contract.json\':continue\n        if rel.parts[0]==\'probes\':\n            repair.reconcile(rel,lambda p:validate_probe(p,config,ch))\n        elif rel.suffix==\'.json\':\n            def json_ok(p):\n                x=json.loads(p.read_text())\n                return isinstance(x,dict) and x.get(\'config_sha256\',ch)==ch\n            repair.reconcile(rel,json_ok)\n        else:\n            # Ancillary locks/debug blobs: copy absent files only; conflicting bytes need inspection.\n            a=local/rel;b=drive/rel\n            if a.exists() and b.exists() and sha(a)!=sha(b):\n                repair.preserve(a);repair.preserve(b);raise ValueError(\'Ancillary file conflict: \'+str(rel))\n            repair.reconcile(rel,lambda p:p.stat().st_size>0)\n    done=len(list((local/\'probes\').glob(\'*.json\')))\n    repair.record(action=\'resume_copy_complete\',config_sha256=ch,cache_chunks=len(markers),saved_probe_files=done)\n    print(\'Saved probe files:\',done,\'/\',len(config[\'tasks\']),flush=True)\n    print(\'Ready for frozen CPU runner; its full cache audit runs before fitting.\',flush=True)\n    return str(repair.backup)\n')
namespace={'__name__':'p5_cpu_resume'}
exec(compile(helper_path.read_text(),str(helper_path),'exec'),namespace)
backup_path=namespace['restore'](ROOT,OUT,PERSIST)
print('복구 기록:',backup_path)

## CPU probe만 실행

GPU 추출은 실행하지 않습니다. 기존 완료 파일을 건너뛰고 남은 probe만 계산합니다. 시작 전 전체 cache를 검증하므로 첫 probe 로그가 나오기까지 시간이 걸릴 수 있습니다. 전체 범위는 `[0,1,2]`로 유지합니다. 이 셀이 중단되면 앞의 안전 재개 셀을 실행하고 다시 이어가세요.


In [ ]:
SEEDS=[0,1,2]
# 실행 실패/중단 때에도 CPU 환경 session 증빙을 Drive로 복사합니다.
try:
    subprocess.run([sys.executable,'-m','interp_v1_4.p5','probes','--root',str(ROOT),
                    '--output',str(OUT),'--persistent',str(PERSIST),'--seeds',*map(str,SEEDS)],check=True)
finally:
    from interp_v1_4.runtime import verified_copy
    for src in (OUT/'sessions').rglob('*'):
        if src.is_file():
            dest=PERSIST/src.relative_to(OUT)
            if dest.exists() and sha(src)!=sha(dest):
                print('session 충돌 보존; 다음 안전 재개 셀에서 검증:',dest)
            else:verified_copy(src,dest)


## 작은 metadata만 회수
CPU probe 도중 중단했거나 완료했을 때 실행할 수 있습니다. 15GB 전체 evidence ZIP을 만들지 않습니다. 결과 ZIP/index를 전달하면 진행률과 오류를 확인할 수 있습니다. 실제 cache는 Drive에 보존합니다.

In [ ]:
transfer_namespace={'__name__':'p5_transfer'}
exec(compile('"""Transport-only P5 multipart export/import; frozen experiment code stays unchanged."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport shutil\nimport zipfile\n\n\ndef sha(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n    return h.hexdigest()\n\n\ndef pack(root,destination,seed=None,limit=512*1024**2):\n    root=Path(root);destination=Path(destination)\n    if destination.resolve().is_relative_to(root.resolve()):raise ValueError(\'Export folder must be outside run folder\')\n    destination.mkdir(parents=True,exist_ok=True)\n    contract=root/\'contract.json\'\n    if not contract.exists():raise ValueError(\'Missing run contract; select P5_r2 folder\')\n    files=[]\n    for p in sorted(root.rglob(\'*\')):\n        if not p.is_file() or p.name.endswith(\'.tmp\'):continue\n        name=p.relative_to(root)\n        is_array=name.parts[0]==\'cache\' and p.suffix==\'.npz\'\n        if seed is None:\n            if not is_array:files.append(p)\n        elif is_array and name.parts[1] in (f\'seed{seed}_trained\',f\'seed{seed}_init\'):\n            marker=p.with_suffix(\'.json\')\n            if not marker.exists():continue\n            info=json.loads(marker.read_text())\n            if sha(p)!=info[\'sha256\']:raise ValueError(\'Corrupt source cache: \'+str(p))\n            files.extend([p,marker])\n    if not files:raise ValueError(\'No completed files for requested scope\')\n    # Each part is an independently usable ZIP, not a byte slice of a 15GB archive.\n    groups=[];current=[];size=0\n    for p in files:\n        n=p.stat().st_size\n        if current and size+n>limit:groups.append(current);current=[];size=0\n        current.append(p);size+=n\n    if current:groups.append(current)\n    scope=\'metadata\' if seed is None else f\'seed{seed}\'\n    index=dict(schema=\'p5-transfer-v1\',scope=scope,contract_sha256=sha(contract),parts=[])\n    for i,group in enumerate(groups):\n        path=destination/f\'p5_{scope}_{i+1:03d}.zip\'\n        inventory={str(p.relative_to(root)):sha(p) for p in group}\n        manifest=dict(schema=\'p5-transfer-part-v1\',contract_sha256=sha(contract),scope=scope,files=inventory)\n        if path.exists():\n            with zipfile.ZipFile(path) as z:\n                if json.loads(z.read(\'transfer_manifest.json\'))!=manifest:raise ValueError(\'Existing part differs; use new export folder\')\n                for name,digest in inventory.items():\n                    h=hashlib.sha256()\n                    with z.open(name) as f:\n                        for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n                    if h.hexdigest()!=digest:raise ValueError(\'Corrupt existing part\')\n        else:\n            tmp=path.with_suffix(\'.zip.tmp\')\n            with zipfile.ZipFile(tmp,\'w\',zipfile.ZIP_DEFLATED,compresslevel=1) as z:\n                for p in group:z.write(p,str(p.relative_to(root)))\n                z.writestr(\'transfer_manifest.json\',json.dumps(manifest))\n            tmp.replace(path)\n        index[\'parts\'].append(dict(name=path.name,bytes=path.stat().st_size,sha256=sha(path),files=len(group)))\n        print(path.name,path.stat().st_size,flush=True)\n    target=destination/f\'p5_{scope}_index.json\'\n    content=json.dumps(index,indent=2)+\'\\n\'\n    if target.exists() and target.read_text()!=content:raise ValueError(\'Index conflict\')\n    target.write_text(content)\n    return index\n\n\ndef receive(archive,output,expected_contract):\n    output=Path(output);output.mkdir(parents=True,exist_ok=True)\n    with zipfile.ZipFile(archive) as z:\n        names=z.namelist();manifest=json.loads(z.read(\'transfer_manifest.json\'))\n        if manifest[\'contract_sha256\']!=expected_contract:raise ValueError(\'Different experiment contract\')\n        if len(names)!=len(set(names)) or set(names)!=set(manifest[\'files\'])|{\'transfer_manifest.json\'}:raise ValueError(\'Archive inventory mismatch\')\n        for name,digest in manifest[\'files\'].items():\n            dest=output/name\n            if not dest.resolve().is_relative_to(output.resolve()):raise ValueError(\'Unsafe archive path\')\n            if dest.exists():\n                if sha(dest)!=digest:raise ValueError(\'Existing file conflict: \'+name)\n                continue\n            dest.parent.mkdir(parents=True,exist_ok=True)\n            tmp=dest.with_suffix(dest.suffix+\'.transfer.tmp\')\n            try:\n                with z.open(name) as src,tmp.open(\'wb\') as dst:shutil.copyfileobj(src,dst,1024*1024)\n                if sha(tmp)!=digest:raise ValueError(\'Checksum failure: \'+name)\n                tmp.replace(dest)\n            finally:\n                if tmp.exists():tmp.unlink()\n    return len(manifest[\'files\'])\n\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();sub=p.add_subparsers(dest=\'action\',required=True)\n    a=sub.add_parser(\'pack\');a.add_argument(\'root\');a.add_argument(\'destination\');a.add_argument(\'--seed\',type=int,choices=[0,1,2])\n    a=sub.add_parser(\'receive\');a.add_argument(\'archive\');a.add_argument(\'output\');a.add_argument(\'--contract\',default=\'experiment_v1_4/p5_r2/contract.json\')\n    args=p.parse_args()\n    if args.action==\'pack\':pack(args.root,args.destination,args.seed)\n    else:print(\'Verified imported files:\',receive(args.archive,args.output,sha(args.contract)))\n','p5_transfer','exec'),transfer_namespace)
EXPORT=PERSIST.parent/('P5_metadata_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S'))
index=transfer_namespace['pack'](PERSIST,EXPORT)
print(json.dumps(index,indent=2))
files.download(str(EXPORT/'p5_metadata_index.json'))

In [ ]:
PART=1
files.download(str(EXPORT/index['parts'][PART-1]['name']))